# LangChain: Models, Prompts and Output Parsers


## Outline

 * Direct API calls to OpenAI
 * API calls through LangChain:
   * Prompts
   * Models
   * Output parsers

## Get your [OpenAI API Key](https://platform.openai.com/account/api-keys)

In [1]:
# pip install openai

In [2]:
import os
from openai import OpenAI

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
client = OpenAI()

## Chat API : OpenAI

Let's start with a direct API calls to OpenAI.

In [3]:
llm_model="gpt-3.5-turbo"

In [4]:
def get_completion(prompt, model=llm_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, 
    )
    return response.choices[0].message.content

In [5]:
get_completion("What is 1+1?")

'1+1 equals 2.'

In [6]:
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls
with smoothie! And to make matters worse, the warranty don't cover the cost of
cleaning up me kitchen. I need yer help right now, matey!
"""

In [7]:
style = """American English
in a calm and respectful tone
"""

In [8]:
prompt = f"""Translate the text that is delimited by triple backticks 
into a style that is {style}. text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English
in a calm and respectful tone
. text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls
with smoothie! And to make matters worse, the warranty don't cover the cost of
cleaning up me kitchen. I need yer help right now, matey!
```



In [9]:
response = get_completion(prompt)
response

"I am really upset that my blender lid flew off and splattered my kitchen walls with smoothie! And to make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help right now, friend."

## Chat API : LangChain

Let's try how we can do the same using LangChain.

In [10]:
# pip install --upgrade langchain

In [11]:
from langchain_openai import ChatOpenAI

In [12]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(temperature=0.0, model=llm_model)
chat

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x11074b250>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x110d10830>, root_client=<openai.OpenAI object at 0x1

### Prompt template

In [13]:
template_string = """Translate the text
that is delimited by triple backticks
into a style that is {style}.
text: ```{text}```
"""

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [15]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text\nthat is delimited by triple backticks\ninto a style that is {style}.\ntext: ```{text}```\n')

In [16]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [17]:
customer_style = """American English
in a calm and respectful tone
"""

In [18]:
customer_email = """
Arrr, I be fuming that me blender lid
flew off and splattered me kitchen walls
with smoothie! And to make matters worse,
the warranty don't cover the cost of
cleaning up me kitchen. I need yer help
right now, matey!
"""

In [19]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [20]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [21]:
print(customer_messages[0])

content="Translate the text\nthat is delimited by triple backticks\ninto a style that is American English\nin a calm and respectful tone\n.\ntext: ```\nArrr, I be fuming that me blender lid\nflew off and splattered me kitchen walls\nwith smoothie! And to make matters worse,\nthe warranty don't cover the cost of\ncleaning up me kitchen. I need yer help\nright now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [22]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)

In [23]:
print(customer_response.content)

I am really frustrated that my blender lid flew off and splattered my kitchen walls with smoothie! And to make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help right now, friend.


In [24]:
service_reply = """Hey there customer, the warranty does not cover
cleaning expenses for your kitchen because it's your fault that
you misused your blender by forgetting to put the lid on before
starting the blender. Tough luck! See ya!
"""

In [25]:
service_style_pirate = """\
a polite tone that speaks in English Pirate
"""

In [26]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text
that is delimited by triple backticks
into a style that is a polite tone that speaks in English Pirate
.
text: ```Hey there customer, the warranty does not cover
cleaning expenses for your kitchen because it's your fault that
you misused your blender by forgetting to put the lid on before
starting the blender. Tough luck! See ya!
```



In [27]:
service_response = chat.invoke(service_messages)
print(service_response.content)

Ahoy there, valued customer! The warranty be not coverin' the cost o' cleanin' yer galley, as it be yer own doin' fer misusin' yer blender by forgettin' to secure the lid afore startin' it. 'Tis a tough break, matey! Fare thee well!


## Output Parsers

Let's start with defining how we would like the LLM output to look like:

In [28]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [29]:
customer_review = """
This leaf blower is pretty amazing.  It has four settings
candle blower, gentle breeze, windy city, and tornado.
It arrived in two days, just in time for my wife's
anniversary present.
I think my wife liked it so much she was speechless.
So far I've been the only one using it, and I've been
using it every other morning to clear the leaves on our lawn.
It's slightly more expensive than the other leaf blowers
out there, but I think it's worth it for the extra features.
"""

review_template = """
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\nFor the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else?\nAnswer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product\nto arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,\nand output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [31]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOpenAI(temperature=0.0, model=llm_model)
response = chat.invoke(messages)
print(response.content)

{
  "gift": true,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there"]
}


In [32]:
type(response.content)

str

In [33]:
# You will get an error by running this line of code 
# because'gift' is not a dictionary
# 'gift' is a string
try:
    response.content.get('gift')
except AttributeError as e:
    print(f"Error: {e}")    

Error: 'str' object has no attribute 'get'


### Parse the LLM output string into a Python dictionary

In [34]:
# pip install "langchain>=1.3" "langchain-openai>=1.4" "langchain-core>=1.5"

In [35]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

In [36]:
class ReviewOutput(BaseModel):
    gift: bool = Field(description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.")
    delivery_days: int = Field(description="How many days did it take for the product to arrive? If not found, output -1.")
    price_value: str = Field(description="Extract any sentences about the value or price, output as a comma separated Python list.")

structured_chat = chat.with_structured_output(ReviewOutput, method="function_calling")
response = structured_chat.invoke(messages)

print(response.gift)
print(response.delivery_days)
print(response.price_value)

False
2
It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


In [37]:
review_template_2 = """\
  For the following text, extract the following information:

  gift: Was the item purchased as a gift for someone else? \
  Answer True if yes, False if not or unknown.

  delivery_days: How many days did it take for the product\
  to arrive? If this information is not found, output -1.

  price_value: Extract any sentences about the value or price,\
  and output them as a comma separated Python list.

  text: {text}
  """

prompt = ChatPromptTemplate.from_template(template=review_template_2)
messages = prompt.format_messages(text=customer_review)

structured_chat = chat.with_structured_output(ReviewOutput, method="function_calling")
response = structured_chat.invoke(messages)

print(response.gift)
print(response.delivery_days)
print(response.price_value)

False
2
It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.
